In [2]:
"""
Files On hold:

Ddata/PMS 360 Employee Report_PMS 360.xls (file corrupted)
Ddata/Probation Confirmation Report.xls (file corrupted)

Files Cleaned:

Performance Goal Report 25-26.xlsx (already exists)
Goal Status Report 2025-26 All.xlsx (already exists)
Historical Ratings and Other Information.xlsx
Performance 360 degree Feedback participants status - All.xlsx
PIP Transaction Report.xls
PMS Q2 25-26 Rating report - all.xlsx
PMS Task Status Report All.xlsx

"""

'\nFiles On hold:\n\nDdata/PMS 360 Employee Report_PMS 360.xls (file corrupted)\nDdata/Probation Confirmation Report.xls (file corrupted)\n\nFiles Cleaned:\n\nPerformance Goal Report 25-26.xlsx (already exists)\nGoal Status Report 2025-26 All.xlsx (already exists)\nHistorical Ratings and Other Information.xlsx\nPerformance 360 degree Feedback participants status - All.xlsx\nPIP Transaction Report.xls\nPMS Q2 25-26 Rating report - all.xlsx\nPMS Task Status Report All.xlsx\n\n'

In [3]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()

MONGO_URI = os.getenv("MONGODB_URI")
if not MONGO_URI:
    raise RuntimeError("MONGO_URI not found in .env")
client = MongoClient(MONGO_URI)
db = client["hr-cleaned"]

In [16]:
import os

folder_path = "C:/Users/SaptarshiBanik/Projects/Tata Play/mongo-rag/src/Ddata"
files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

print("Files in the directory:")
for file in files:
    print(f"Ddata/{file}")

Files in the directory:
Ddata/8.Leave Transaction With Balance Report_Leave Transaction With Balance Report (3).xlsx
Ddata/Goal Detail Report.xlsx
Ddata/Goal Status Report 2025-26 All.xlsx
Ddata/Historical Ratings and Other Information.xlsx
Ddata/Output1.xls
Ddata/Performance 360 degree Feedback participants status - All.xlsx
Ddata/Performance Goal Report 25-26.xlsx
Ddata/PIP Transaction Report.xls
Ddata/PMS 360 Employee Report_PMS 360.xls
Ddata/PMS Q2 25-26 Rating report - all.xlsx
Ddata/PMS Task Status Report All.xlsx
Ddata/Probation Confirmation Report.xls


In [17]:
from datetime import datetime as dt
import pandas as pd


In [18]:
def parse_date(value):
    if pd.isna(value) or value == "":
        return None

    if isinstance(value, (dt, pd.Timestamp)):
        return value.to_pydatetime() if hasattr(value, "to_pydatetime") else value

    val = str(value).strip()

    date_formats = [
        "%Y-%m-%d",
        "%d-%m-%Y",
        "%d/%m/%Y",
        "%Y/%m/%d",
        "%d-%b-%Y",                
        "%d-%b-%Y".upper(),         
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%dT%H:%M:%S.%fZ",
        "%Y-%m-%dT%H:%M:%S.%f%z"    
    ]

    for fmt in date_formats:
        try:
            return dt.strptime(val, fmt)
        except:
            pass

    return None


In [19]:
def normalize_doc_generic(raw_doc, fields, key_renames, numeric_fields, date_fields):
    cleaned = {}

    for field in fields:
        old_key = field
        new_key = key_renames.get(field, field)

        value = raw_doc.get(old_key)

        # Missing value
        if value is None:
            if new_key in numeric_fields:
                cleaned[new_key] = None
            elif new_key in date_fields:
                cleaned[new_key] = None  
            else:
                cleaned[new_key] = "NA"
            continue

        # Date fields
        if new_key in date_fields:
            # If already MongoDB extended JSON date, keep as is
            if isinstance(value, dict) and "$date" in value:
                cleaned[new_key] = value
            else:
                cleaned[new_key] = parse_date(value)
            continue

        # Numeric fields
        if new_key in numeric_fields:
            try:
                cleaned[new_key] = int(value)
            except Exception:
                cleaned[new_key] = None
            continue

        # Default: treat as string field
        value_str = str(value).strip()
        value_str.replace("...","")
        value_str.strip()
        cleaned[new_key] = value_str if value_str else "NA"


    return cleaned



## Ddata/Historical Ratings and Other Information.xlsx

In [31]:
path="Ddata/Historical Ratings and Other Information.xlsx"
collection="historical_ratings_and-other_information"
df=pd.read_excel(path)

In [10]:
df.columns

Index(['Employee Code', 'FIRST NAME', 'LAST NAME', 'GRADE', 'Grade Level',
       'Designation', 'DEPARTMENT', 'SUB-DEPT', 'Location', 'Office', 'Region',
       'DOJ', 'Assignment Status Type', 'Final Rating  20-21',
       'Final Rating  21-22', 'Final Rating  22-23', 'Final Rating  23-24',
       'Last Promotion date', 'Rating 24-25', 'Experience Prior to Tata Play',
       'Tata Play Experience', 'Total Yrs Exp', 'Previous company', 'MT Batch',
       'Highest Qualification', 'University Name'],
      dtype='object')

In [16]:
FIELDS = [
    "Employee Code",
    "FIRST NAME",
    "LAST NAME",
    "GRADE",
    "Grade Level",
    "Designation",
    "DEPARTMENT",
    "SUB-DEPT",
    "Location",
    "Office",
    "Region",
    "DOJ",
    "Assignment Status Type",
    "Final Rating  20-21",
    "Final Rating  21-22",
    "Final Rating  22-23",
    "Final Rating  23-24",
    "Last Promotion date",
    "Rating 24-25",
    "Experience Prior to Tata Play",
    "Tata Play Experience",
    "Total Yrs Exp",
    "Previous company",
    "MT Batch",
    "Highest Qualification",
    "University Name"
]
KEY_RENAMES = {
    "Employee Code": "employee code",
    "FIRST NAME": "first name",
    "LAST NAME": "last name",
    "GRADE": "grade",
    "Grade Level": "grade level",
    "Designation": "designation",
    "DEPARTMENT": "department",
    "SUB-DEPT": "sub department",
    "Location": "location",
    "Office": "office",
    "Region": "region",
    "DOJ": "date of joining",
    "Assignment Status Type": "assignment status type",
    "Final Rating  20-21": "final rating 20-21",
    "Final Rating  21-22": "final rating 21-22",
    "Final Rating  22-23": "final rating 22-23",
    "Final Rating  23-24": "final rating 23-24",
    "Last Promotion date": "last promotion date",
    "Rating 24-25": "rating 24-25",
    "Experience Prior to Tata Play": "experience prior to tata play",
    "Tata Play Experience": "tata play experience",
    "Total Yrs Exp": "total years experience",
    "Previous company": "previous company",
    "MT Batch": "mt batch",
    "Highest Qualification": "highest qualification",
    "University Name": "university name"
}
NUMERIC_FIELDS = [
    "employee code",
    "final rating 20-21",
    "final rating 21-22",
    "final rating 22-23",
    "final rating 23-24",
    "rating 24-25",
    "experience prior to tata play",
    "tata play experience",
    "total years experience",
    "mt batch"
]
DATE_FIELDS = [
    "date of joining",
    "last promotion date"
]


In [33]:
raw_docs = []

# Convert each DF row → raw dict
for idx, row in df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [34]:
normalized_docs

[{'employee code': 1004,
  'first name': 'Gyanendra',
  'last name': 'Singh',
  'grade': 'M5',
  'grade level': 'M5a',
  'designation': 'Assistant Manager - Quality',
  'department': 'Technology',
  'sub department': 'Field Engineering and Audit',
  'location': 'Lucknow',
  'office': 'Lucknow',
  'region': 'North',
  'date of joining': datetime.datetime(2006, 4, 3, 0, 0),
  'assignment status type': 'ACTIVE',
  'final rating 20-21': 3,
  'final rating 21-22': 3,
  'final rating 22-23': 3,
  'final rating 23-24': 3,
  'last promotion date': None,
  'rating 24-25': 3,
  'experience prior to tata play': 13,
  'tata play experience': 19,
  'total years experience': 32,
  'previous company': 'Adonis Electronics Pvt. Ltd.',
  'mt batch': None,
  'highest qualification': 'Diploma (Electrical Engineering)',
  'university name': 'Board of Technical Education, U.P.'},
 {'employee code': 1034,
  'first name': 'Monoranjon',
  'last name': 'Dutta',
  'grade': 'M4',
  'grade level': 'M4a',
  'design

In [36]:
dst = db[collection]
result = dst.insert_many(normalized_docs)

## Ddata/Performance 360 degree Feedback participants status - All.xlsx

In [38]:
path="Ddata/Performance 360 degree Feedback participants status - All.xlsx"
collection="performance_360_degree_feedback_participants_status_all"
df=pd.read_excel(path)

In [39]:
df.columns

Index(['Person Number', 'Name', 'Parent Grade', 'Business Unit Name',
       'Parent Department', 'Department Name', 'Performance Document Name',
       'Participant Name', 'Role', 'Role Status', 'Participation Status'],
      dtype='object')

In [40]:
FIELDS = [
    "Person Number",
    "Name",
    "Parent Grade",
    "Business Unit Name",
    "Parent Department",
    "Department Name",
    "Performance Document Name",
    "Participant Name",
    "Role",
    "Role Status",
    "Participation Status"
]
KEY_RENAMES = {
    "Person Number": "employee code",
    "Name": "employee name",
    "Parent Grade": "grade",
    "Business Unit Name": "region",
    "Parent Department": "department",
    "Department Name": "sub department",
    "Performance Document Name": "performance document name",
    "Participant Name": "participant name",
    "Role": "role",
    "Role Status": "role status",
    "Participation Status": "participation status"
}
NUMERIC_FIELDS = [
    "employee code"
]
DATE_FIELDS = []


In [41]:
raw_docs = []

# Convert each DF row → raw dict
for idx, row in df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [42]:
normalized_docs

[{'employee code': 107,
  'employee name': 'Chirag P. Thakar',
  'grade': 'M3',
  'region': 'Corporate',
  'department': 'Finance',
  'sub department': 'Finance - General',
  'performance document name': 'Anchor Competency 360 Feedback - M3 - 2025 (Individual Contributor)',
  'participant name': 'Avinash Kesherwani',
  'role': 'Manager',
  'role status': 'Active',
  'participation status': 'Not Started'},
 {'employee code': 107,
  'employee name': 'Chirag P. Thakar',
  'grade': 'M3',
  'region': 'Corporate',
  'department': 'Finance',
  'sub department': 'Finance - General',
  'performance document name': 'Anchor Competency 360 Feedback - M3 - 2025 (Individual Contributor)',
  'participant name': 'Chirag P. Thakar',
  'role': 'Worker',
  'role status': 'Active',
  'participation status': 'Not Started'},
 {'employee code': 107,
  'employee name': 'Chirag P. Thakar',
  'grade': 'M3',
  'region': 'Corporate',
  'department': 'Finance',
  'sub department': 'Finance - General',
  'performan

In [43]:
dst = db[collection]
result = dst.insert_many(normalized_docs)

## Ddata/PIP Transaction Report.xls

In [59]:


path="Ddata/PIP Transaction Report.xls"
collection="pip_transaction_report"
df=pd.read_excel(path)

In [60]:
df['Assigned To'] = df['Assigned To'].fillna('')

In [61]:
df.columns

Index(['PERSON_NUMBER', 'First_Name', 'Last_Name', 'Employee_Email_ID',
       'Parent_Grade', 'Sub_grade', 'Designation', 'SUB_DEPARTMENT',
       'Office_Location', 'Region', 'DOJ', 'Manager_Number', 'MAN_EMAIL',
       'PIP Submitted Date', 'PIP Completion Date', 'RHR_NAME', 'RHR_Number',
       'RHR_EMAIL', 'Reviewer_NAME', 'Reviewer_Number', 'Reviewer_EMAIL',
       'PIP Submitted Date.1', 'PIP Completion date', 'Assigned Date',
       'Assigned To', '\t\nTask status', 'PIP Doc status'],
      dtype='object')

In [49]:
# removed 2 duplicate columns
FIELDS = [
    "PERSON_NUMBER",
    "First_Name",
    "Last_Name",
    "Employee_Email_ID",
    "Parent_Grade",
    "Sub_grade",
    "Designation",
    "SUB_DEPARTMENT",
    "Office_Location",
    "Region",
    "DOJ",
    "Manager_Number",
    "MAN_EMAIL",
    "PIP Submitted Date",
    "PIP Completion Date",
    "RHR_NAME",
    "RHR_Number",
    "RHR_EMAIL",
    "Reviewer_NAME",
    "Reviewer_Number",
    "Reviewer_EMAIL",
    "Assigned Date",
    "Assigned To",
    "\t\nTask status",
    "PIP Doc status"
]

KEY_RENAMES = {
    "PERSON_NUMBER": "employee code",
    "First_Name": "first name",
    "Last_Name": "last name",
    "Employee_Email_ID": "employee email",
    "Parent_Grade": "grade",
    "Sub_grade": "grade level",
    "Designation": "designation",
    "SUB_DEPARTMENT": "sub department",
    "Office_Location": "office",
    "Region": "region",
    "DOJ": "date of joining",
    "Manager_Number": "manager employee code",
    "MAN_EMAIL": "manager email",
    "PIP Submitted Date": "pip submitted date",
    "PIP Completion Date": "pip completion date",
    "RHR_NAME": "rhr name",
    "RHR_Number": "rhr employee code",
    "RHR_EMAIL": "rhr email",
    "Reviewer_NAME": "reviewer name",
    "Reviewer_Number": "reviewer employee code",
    "Reviewer_EMAIL": "reviewer email",
    "Assigned Date": "assigned date",
    "Assigned To": "assigned to",
    "\t\nTask status": "task status",
    "PIP Doc status": "pip document status"
}

NUMERIC_FIELDS = [
    "employee code",
    "manager employee code",
    "rhr employee code",
    "reviewer employee code"
]

DATE_FIELDS = [
    "date of joining",
    "pip submitted date",
    "pip completion date",
    "assigned date"
]


In [62]:
raw_docs = []

# Convert each DF row → raw dict
for idx, row in df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [63]:
normalized_docs

[{'employee code': 3159,
  'first name': 'Kalpesh Kumar',
  'last name': 'Sagar',
  'employee email': 'kalpeshkumar.sagar@tataplay.com',
  'grade': 'M5',
  'grade level': 'M5b',
  'designation': 'Senior Executive - Field Service Delivery',
  'sub department': 'Field Service Delivery - General',
  'office': 'Ahmedabad',
  'region': 'West',
  'date of joining': datetime.datetime(2011, 7, 4, 0, 0),
  'manager employee code': 4735,
  'manager email': 'Sandesh.Patel@tataplay.com',
  'pip submitted date': datetime.datetime(2024, 11, 28, 0, 0),
  'pip completion date': datetime.datetime(2024, 8, 20, 0, 0),
  'rhr name': 'Pramatesh V. Kumar',
  'rhr employee code': 871,
  'rhr email': 'pramateshk@tataplay.com',
  'reviewer name': 'Vikas Chugh',
  'reviewer employee code': 299,
  'reviewer email': 'vikasc@tataplay.com',
  'assigned date': None,
  'assigned to': 'NA',
  'task status': 'COMPLETE',
  'pip document status': 'APPROVED'},
 {'employee code': 4015,
  'first name': 'Vivek',
  'last name

In [65]:
dst = db[collection]
result = dst.insert_many(normalized_docs)

## Ddata/PMS 360 Employee Report_PMS 360.xls

In [74]:
path="Ddata/PMS 360 Employee Report_PMS 360.xls"
collection="pms_360_employee_report_pms_360"
df=pd.read_excel(path,engine="xlrd")

XLRDError: Unsupported format, or corrupt file: Expected BOF record; found b'Date: Tu'

## Ddata/PMS Q2 25-26 Rating report - all.xlsx

In [6]:
path="../Ddata/PMS Q2 25-26 Rating report - all.xlsx"
collection="pms_q2_25_26_rating_report_all"
df=pd.read_excel(path,skiprows=2)

In [7]:
df.columns

Index(['Employee Person Number', 'Employee Name', 'Employee E-Mail Address',
       'Manager Person Number', 'Manager Name', 'Manager E-Mail Address',
       'Business Unit Name', 'Department', 'Parent Department', 'Grade',
       'Parent Grade', 'Job', 'Location Name', 'Performance Document Name',
       'Overall Manager Rating', 'Overall Employee Rating',
       'Performance Document Status'],
      dtype='object')

In [81]:
df

,Employee Person Number,Employee Name,Employee E-Mail Address,Manager Person Number,Manager Name,Manager E-Mail Address,Business Unit Name,Department,Parent Department,Grade,Parent Grade,Job,Location Name,Performance Document Name,Overall Manager Rating,Overall Employee Rating,Performance Document Status
0,1004.0,Gyanendra Singh,gyanendras@tataplay.com,2430.0,Vishal Sethi,vishal.sethi@tataplay.com,North,Field Engineering and Audit,Technology,M5a,M5,Assistant Manager - Quality,Lucknow,2025-26 – Q2/H1 (Mid Year),3.0,NaN,Completed
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,Completed
2,1034.0,Monoranjon Dutta,manoranjand@tataplay.com,4922.0,Joydeep Roy,Joydeep.Roy@tataplay.com,East,Field Service Delivery - General,Field Service Delivery,M4a,M4,Senior Manager - Field Service Delivery,Kolkata,2025-26 – Q2/H1 (Mid Year),3.0,NaN,Completed
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,Completed
4,1045.0,Metilda Molly Thomas,mollyt@tataplay.com,5922.0,Nipun Sharma,Nipun.Sharma@tataplay.com,South,Customer Operations - General,Customer Operations,M4b,M4,Manager - Customer Operations,Hebbal,2025-26 – Q2/H1 (Mid Year),3.0,NaN,Approved
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2478,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,Approved
2479,950.0,Dharmesh Shah,dharmeshs@tataplay.com,941.0,Jayesh Joshi,jayeshj@tataplay.com,West,Field Service Delivery - General,Field Service Delivery,M4b,M4,Manager - Field Service Delivery,Ahmedabad,2025-26 – Q2/H1 (Mid Year),2.0,NaN,Completed
2480,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,Completed
2481,997.0,Ramesh Rai,rameshr@tataplay.com,6471.0,Sadashiv Sharma,Sadashiv.Sharma@tataplay.com,North,Trade Sales,Sales,M5a,M5,Assistant Manager - Trade Sales,Noida,2025-26 – Q2/H1 (Mid Year),2.0,NaN,Completed


In [ ]:
import pandas as pd

col = "Performance Document Status"

# List to store incorrect pairs
mismatches = []

# Iterate in steps of 2: (0,1), (2,3), (4,5), ...
for i in range(0, len(df), 2):
    if i + 1 < len(df):  # ensure pair exists
        v1 = df.loc[i, col]
        v2 = df.loc[i+1, col]

        if pd.isna(v1) or pd.isna(v2):
            continue  # skip if data missing (optional)

        if v1 != v2:
            print()
            mismatches.append((i, i+1, v1, v2))

# Final report
if mismatches:
    print("Mismatch found in the following row pairs:")
    for r1, r2, v1, v2 in mismatches:
        print(f"Rows {r1} & {r2}: '{v1}' != '{v2}'")
else:
    print("All row pairs match for Performance Document Status.")


Mismatch found in the following row pairs:
Rows 208 & 209: 'In progress' != 'Completed'
Rows 220 & 221: 'Completed' != 'Approved'
Rows 224 & 225: 'Approved' != 'Completed'
Rows 244 & 245: 'Completed' != 'In progress'
Rows 246 & 247: 'In progress' != 'Completed'
Rows 260 & 261: 'Completed' != 'In progress'
Rows 262 & 263: 'In progress' != 'Completed'
Rows 276 & 277: 'Completed' != 'Approved'
Rows 278 & 279: 'Approved' != 'Completed'
Rows 288 & 289: 'Completed' != 'In progress'
Rows 290 & 291: 'In progress' != 'Completed'
Rows 294 & 295: 'Completed' != 'Approved'
Rows 296 & 297: 'Approved' != 'Completed'
Rows 298 & 299: 'Completed' != 'In progress'
Rows 300 & 301: 'In progress' != 'Completed'
Rows 308 & 309: 'Completed' != 'Approved'
Rows 310 & 311: 'Approved' != 'Completed'
Rows 312 & 313: 'Completed' != 'Submitted'
Rows 314 & 315: 'Submitted' != 'Completed'
Rows 318 & 319: 'Completed' != 'In progress'
Rows 320 & 321: 'In progress' != 'Completed'
Rows 330 & 331: 'Completed' != 'Approved

In [91]:

row_counts = df.count(axis=1)
row_counts

# A typical main row in your screenshot has ~12–16 filled columns
# A continuation row has 1–3 filled columns
MAIN_MIN = 10          # consider row main if >= this many non-null columns
CONT_MAX = 4          # consider continuation if <= this many non-null columns
classifications = []   # will store: "main", "cont", "isolated", "unknown"

for i in range(len(df)):
    cnt = row_counts[i]

    if cnt >= MAIN_MIN:
        classifications.append("main")
    elif cnt <= CONT_MAX:
        classifications.append("cont")
    else:
        classifications.append("unknown")
merged_blocks = []
i = 0

while i < len(df):
    if classifications[i] == "main":
        block = [i]

        # collect all continuation rows after it
        j = i + 1
        while j < len(df) and classifications[j] == "cont":
            block.append(j)
            j += 1

        merged_blocks.append(block)
        i = j

    elif classifications[i] == "cont":
        merged_blocks.append([i])   # stray continuation (bad data)
        i += 1

    else:
        merged_blocks.append([i])   # unknown irregular row
        i += 1

print("\nDetected Groups:")
for block in merged_blocks:
    row_idx = block[0]
    emp_num = df.loc[row_idx, "Employee Person Number"]
    if len(block) == 1:
        print(f"Row {row_idx} (single row) — Employee Person Number: {emp_num}")
    else:
        print(f"Rows {block} form a merged multi-row block")
        row_idx1 = block[0]
        row_idx2 = block[1]
        s1 = df.loc[row_idx1, "Performance Document Status"]
        s2= df.loc[row_idx2, "Performance Document Status"]
        if s1 !=s2:
            print("Double Problem",emp_num)




Detected Groups:
Rows [0, 1] form a merged multi-row block
Rows [2, 3] form a merged multi-row block
Rows [4, 5] form a merged multi-row block
Rows [6, 7] form a merged multi-row block
Rows [8, 9] form a merged multi-row block
Rows [10, 11] form a merged multi-row block
Rows [12, 13] form a merged multi-row block
Rows [14, 15] form a merged multi-row block
Rows [16, 17] form a merged multi-row block
Rows [18, 19] form a merged multi-row block
Rows [20, 21] form a merged multi-row block
Rows [22, 23] form a merged multi-row block
Rows [24, 25] form a merged multi-row block
Rows [26, 27] form a merged multi-row block
Rows [28, 29] form a merged multi-row block
Rows [30, 31] form a merged multi-row block
Rows [32, 33] form a merged multi-row block
Rows [34, 35] form a merged multi-row block
Rows [36, 37] form a merged multi-row block
Rows [38, 39] form a merged multi-row block
Rows [40, 41] form a merged multi-row block
Rows [42, 43] form a merged multi-row block
Rows [44, 45] form a mer

In [93]:
flattened_rows = []
cols = df.columns
for block in merged_blocks:
    # Take the first (main) row of the block
    main_idx = block[0]
    main_row = df.loc[main_idx].copy()

    # If block has more than one row, merge continuation rows
    if len(block) > 1:
        for cont_idx in block[1:]:
            cont_row = df.loc[cont_idx]

            # For each column, if continuation has value, override main
            for col in cols:
                if pd.notna(cont_row[col]):
                    main_row[col] = cont_row[col]

    # Store the assembled row
    flattened_rows.append(main_row)


In [94]:
flattened_df = pd.DataFrame(flattened_rows, columns=cols)
flattened_df.reset_index(drop=True, inplace=True)


In [95]:
flattened_df

,Employee Person Number,Employee Name,Employee E-Mail Address,Manager Person Number,Manager Name,Manager E-Mail Address,Business Unit Name,Department,Parent Department,Grade,Parent Grade,Job,Location Name,Performance Document Name,Overall Manager Rating,Overall Employee Rating,Performance Document Status
0,1004.0,Gyanendra Singh,gyanendras@tataplay.com,2430.0,Vishal Sethi,vishal.sethi@tataplay.com,North,Field Engineering and Audit,Technology,M5a,M5,Assistant Manager - Quality,Lucknow,2025-26 – Q2/H1 (Mid Year),3.0,4.0,Completed
1,1034.0,Monoranjon Dutta,manoranjand@tataplay.com,4922.0,Joydeep Roy,Joydeep.Roy@tataplay.com,East,Field Service Delivery - General,Field Service Delivery,M4a,M4,Senior Manager - Field Service Delivery,Kolkata,2025-26 – Q2/H1 (Mid Year),3.0,3.0,Completed
2,1045.0,Metilda Molly Thomas,mollyt@tataplay.com,5922.0,Nipun Sharma,Nipun.Sharma@tataplay.com,South,Customer Operations - General,Customer Operations,M4b,M4,Manager - Customer Operations,Hebbal,2025-26 – Q2/H1 (Mid Year),3.0,3.0,Approved
3,1060.0,Thangudu Srinivasa Rao,srinivasar@tataplay.com,4127.0,Ashoka B,ashoka.b@tataplay.com,South,Field Service Delivery - General,Field Service Delivery,M4b,M4,Manager - Field Service Delivery,Hebbal,2025-26 – Q2/H1 (Mid Year),4.0,4.0,Completed
4,107.0,Chirag P. Thakar,chiragt@tataplay.com,2747.0,Avinash Kesherwani,avinash.kesherwani@tataplay.com,Corporate,Finance - General,Finance,M3b,M3,Assistant General Manager - Finance,Corporate,2025-26 – Q2/H1 (Mid Year),3.0,4.0,Completed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1254,920.0,Madhu Sharma,madhus@tataplay.com,1480.0,Sanjay Mittal,sanjaymi@tataplay.com,North,Field Service Delivery Support,Field Service Delivery,M4b,M4,Manager - Field Service Delivery Support,Noida,2025-26 – Q2/H1 (Mid Year),2.0,3.0,Approved
1255,926.0,Harihar Nath Vishwakarma,hariharnathv@tataplay.com,299.0,Vikas Chugh,vikasc@tataplay.com,Corporate,MIS & Reporting-Sales,Sales,M4b,M4,Manager - MIS & Reporting-Sales,Corporate,2025-26 – Q2/H1 (Mid Year),3.0,3.0,Approved
1256,941.0,Jayesh Joshi,jayeshj@tataplay.com,299.0,Vikas Chugh,vikasc@tataplay.com,West,Field Service Delivery - General,Field Service Delivery,M3a,M3,General Manager - Field Service Delivery,Ahmedabad,2025-26 – Q2/H1 (Mid Year),3.0,3.0,Approved
1257,950.0,Dharmesh Shah,dharmeshs@tataplay.com,941.0,Jayesh Joshi,jayeshj@tataplay.com,West,Field Service Delivery - General,Field Service Delivery,M4b,M4,Manager - Field Service Delivery,Ahmedabad,2025-26 – Q2/H1 (Mid Year),2.0,3.0,Completed


In [97]:
rows_with_nulls = flattened_df[df.isnull().any(axis=1)]
print("Rows containing NULL values:\n")
print(rows_with_nulls)


Rows containing NULL values:

      Employee Person Number             Employee Name  \
0                     1004.0           Gyanendra Singh   
1                     1034.0          Monoranjon Dutta   
2                     1045.0      Metilda Molly Thomas   
3                     1060.0    Thangudu Srinivasa Rao   
4                      107.0          Chirag P. Thakar   
...                      ...                       ...   
1254                   920.0              Madhu Sharma   
1255                   926.0  Harihar Nath Vishwakarma   
1256                   941.0              Jayesh Joshi   
1257                   950.0             Dharmesh Shah   
1258                   997.0                Ramesh Rai   

        Employee E-Mail Address  Manager Person Number        Manager Name  \
0       gyanendras@tataplay.com                 2430.0        Vishal Sethi   
1      manoranjand@tataplay.com                 4922.0         Joydeep Roy   
2           mollyt@tataplay.com        

C:\Users\SaptarshiBanik\AppData\Local\Temp\ipykernel_14720\3470424408.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  rows_with_nulls = flattened_df[df.isnull().any(axis=1)]


In [98]:
rows_with_nulls

,Employee Person Number,Employee Name,Employee E-Mail Address,Manager Person Number,Manager Name,Manager E-Mail Address,Business Unit Name,Department,Parent Department,Grade,Parent Grade,Job,Location Name,Performance Document Name,Overall Manager Rating,Overall Employee Rating,Performance Document Status
0,1004.0,Gyanendra Singh,gyanendras@tataplay.com,2430.0,Vishal Sethi,vishal.sethi@tataplay.com,North,Field Engineering and Audit,Technology,M5a,M5,Assistant Manager - Quality,Lucknow,2025-26 – Q2/H1 (Mid Year),3.0,4.0,Completed
1,1034.0,Monoranjon Dutta,manoranjand@tataplay.com,4922.0,Joydeep Roy,Joydeep.Roy@tataplay.com,East,Field Service Delivery - General,Field Service Delivery,M4a,M4,Senior Manager - Field Service Delivery,Kolkata,2025-26 – Q2/H1 (Mid Year),3.0,3.0,Completed
2,1045.0,Metilda Molly Thomas,mollyt@tataplay.com,5922.0,Nipun Sharma,Nipun.Sharma@tataplay.com,South,Customer Operations - General,Customer Operations,M4b,M4,Manager - Customer Operations,Hebbal,2025-26 – Q2/H1 (Mid Year),3.0,3.0,Approved
3,1060.0,Thangudu Srinivasa Rao,srinivasar@tataplay.com,4127.0,Ashoka B,ashoka.b@tataplay.com,South,Field Service Delivery - General,Field Service Delivery,M4b,M4,Manager - Field Service Delivery,Hebbal,2025-26 – Q2/H1 (Mid Year),4.0,4.0,Completed
4,107.0,Chirag P. Thakar,chiragt@tataplay.com,2747.0,Avinash Kesherwani,avinash.kesherwani@tataplay.com,Corporate,Finance - General,Finance,M3b,M3,Assistant General Manager - Finance,Corporate,2025-26 – Q2/H1 (Mid Year),3.0,4.0,Completed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1254,920.0,Madhu Sharma,madhus@tataplay.com,1480.0,Sanjay Mittal,sanjaymi@tataplay.com,North,Field Service Delivery Support,Field Service Delivery,M4b,M4,Manager - Field Service Delivery Support,Noida,2025-26 – Q2/H1 (Mid Year),2.0,3.0,Approved
1255,926.0,Harihar Nath Vishwakarma,hariharnathv@tataplay.com,299.0,Vikas Chugh,vikasc@tataplay.com,Corporate,MIS & Reporting-Sales,Sales,M4b,M4,Manager - MIS & Reporting-Sales,Corporate,2025-26 – Q2/H1 (Mid Year),3.0,3.0,Approved
1256,941.0,Jayesh Joshi,jayeshj@tataplay.com,299.0,Vikas Chugh,vikasc@tataplay.com,West,Field Service Delivery - General,Field Service Delivery,M3a,M3,General Manager - Field Service Delivery,Ahmedabad,2025-26 – Q2/H1 (Mid Year),3.0,3.0,Approved
1257,950.0,Dharmesh Shah,dharmeshs@tataplay.com,941.0,Jayesh Joshi,jayeshj@tataplay.com,West,Field Service Delivery - General,Field Service Delivery,M4b,M4,Manager - Field Service Delivery,Ahmedabad,2025-26 – Q2/H1 (Mid Year),2.0,3.0,Completed


In [103]:
import numpy as np

def is_iterable_but_not_string(x):
    return isinstance(x, (list, tuple, set, np.ndarray))
def looks_like_multivalue_string(x):
    if not isinstance(x, str):
        return False
    return ("," in x) or (";" in x) or ("|" in x)

mask_listlike = df.map(is_iterable_but_not_string)
rows_with_list_objects = df[mask_listlike.any(axis=1)]

mask_multi_str = df.map(looks_like_multivalue_string)
rows_with_multi_string = df[mask_multi_str.any(axis=1)]

combined_mask = mask_listlike | mask_multi_str
rows_with_any_multivalue = df[combined_mask.any(axis=1)]

print("Rows with ANY multi-value content (object or string):\n")
rows_with_any_multivalue

Rows with ANY multi-value content (object or string):



,Employee Person Number,Employee Name,Employee E-Mail Address,Manager Person Number,Manager Name,Manager E-Mail Address,Business Unit Name,Department,Parent Department,Grade,Parent Grade,Job,Location Name,Performance Document Name,Overall Manager Rating,Overall Employee Rating,Performance Document Status
374,299.0,Vikas Chugh,vikasc@tataplay.com,293.0,Bhagat Bisht,bishtb@tataplay.com,West,Field Service Delivery - General,Field Service Delivery,M2a,M2,Senior Vice President - Sales and Field Servic...,Corporate,2025-26 – Q2/H1 (Mid Year),NaN,3.0,In progress


In [105]:
flattened_df.columns

Index(['Employee Person Number', 'Employee Name', 'Employee E-Mail Address',
       'Manager Person Number', 'Manager Name', 'Manager E-Mail Address',
       'Business Unit Name', 'Department', 'Parent Department', 'Grade',
       'Parent Grade', 'Job', 'Location Name', 'Performance Document Name',
       'Overall Manager Rating', 'Overall Employee Rating',
       'Performance Document Status'],
      dtype='object')

In [111]:
FIELDS = [
    "Employee Person Number",
    "Employee Name",
    "Employee E-Mail Address",
    "Manager Person Number",
    "Manager Name",
    "Manager E-Mail Address",
    "Business Unit Name",
    "Department",
    "Parent Department",
    "Grade",
    "Parent Grade",
    "Job",
    "Location Name",
    "Performance Document Name",
    "Overall Manager Rating",
    "Overall Employee Rating",
    "Performance Document Status"
]
KEY_RENAMES = {
    "Employee Person Number": "employee code",
    "Employee Name": "employee name",
    "Employee E-Mail Address": "email",

    "Manager Person Number": "manager employee code",
    "Manager Name": "manager name",
    "Manager E-Mail Address": "manager email",

    "Business Unit Name": "region",
    "Department": "sub department",
    "Parent Department": "department",
    
    "Grade": "grade level",
    "Parent Grade": "grade",

    "Job": "designation",

    "Location Name": "location",
    "Performance Document Name": "performance document name",
    "Overall Manager Rating": "overall manager rating",
    "Overall Employee Rating": "overall employee rating",
    "Performance Document Status": "performance document status"
}
NUMERIC_FIELDS = [
    "employee code",
    "manager employee code",
    "overall manager rating",
    "overall employee rating"
]
DATE_FIELDS = []


In [112]:
raw_docs = []

# Convert each DF row → raw dict
for idx, row in flattened_df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [113]:
normalized_docs

[{'employee code': 1004,
  'employee name': 'Gyanendra Singh',
  'email': 'gyanendras@tataplay.com',
  'manager employee code': 2430,
  'manager name': 'Vishal Sethi',
  'manager email': 'vishal.sethi@tataplay.com',
  'region': 'North',
  'sub department': 'Field Engineering and Audit',
  'department': 'Technology',
  'grade level': 'M5a',
  'grade': 'M5',
  'designation': 'Assistant Manager - Quality',
  'location': 'Lucknow',
  'performance document name': '2025-26 – Q2/H1 (Mid Year)',
  'overall manager rating': 3,
  'overall employee rating': 4,
  'performance document status': 'Completed'},
 {'employee code': 1034,
  'employee name': 'Monoranjon Dutta',
  'email': 'manoranjand@tataplay.com',
  'manager employee code': 4922,
  'manager name': 'Joydeep Roy',
  'manager email': 'Joydeep.Roy@tataplay.com',
  'region': 'East',
  'sub department': 'Field Service Delivery - General',
  'department': 'Field Service Delivery',
  'grade level': 'M4a',
  'grade': 'M4',
  'designation': 'Seni

In [114]:
dst = db[collection]
result = dst.insert_many(normalized_docs)

## Ddata/PMS Task Status Report All.xlsx

In [117]:
path="Ddata/PMS Task Status Report All.xlsx"
collection="pms_task_status_report_all"
df=pd.read_excel(path)

In [118]:
df.columns

Index(['Perosn Number', 'Name', 'Employee Email ID', 'Manager Person Number',
       'Manager Name', 'Manager Email Address', 'Reviewer Number',
       'Reviewer Name', 'Business Unit', 'Parent Department', 'Sub Department',
       'Grade', 'Sub Grade', 'Employee Evaluation',
       'Employee Evaluation Status', 'Manager Evaluation',
       'Manager Evaluation Status', 'Initiate Approval',
       'Initiate Approval Status', 'Share Document', 'Share Document Status',
       'Final Status'],
      dtype='object')

In [119]:
FIELDS = [
    "Perosn Number",
    "Name",
    "Employee Email ID",
    "Manager Person Number",
    "Manager Name",
    "Manager Email Address",
    "Reviewer Number",
    "Reviewer Name",
    "Business Unit",
    "Parent Department",
    "Sub Department",
    "Grade",
    "Sub Grade",
    "Employee Evaluation",
    "Employee Evaluation Status",
    "Manager Evaluation",
    "Manager Evaluation Status",
    "Initiate Approval",
    "Initiate Approval Status",
    "Share Document",
    "Share Document Status",
    "Final Status"
]

KEY_RENAMES = {
    "Perosn Number": "employee code",
    "Name": "employee name",
    "Employee Email ID": "email",

    "Manager Person Number": "manager employee code",
    "Manager Name": "manager name",
    "Manager Email Address": "manager email",

    "Reviewer Number": "reviewer employee code",
    "Reviewer Name": "reviewer name",

    "Business Unit": "region",
    "Parent Department": "department",
    "Sub Department": "sub department",

    "Grade": "grade",
    "Sub Grade": "grade level",

    "Employee Evaluation": "employee evaluation",
    "Employee Evaluation Status": "employee evaluation status",

    "Manager Evaluation": "manager evaluation",
    "Manager Evaluation Status": "manager evaluation status",

    "Initiate Approval": "initiate approval",
    "Initiate Approval Status": "initiate approval status",

    "Share Document": "share document",
    "Share Document Status": "share document status",

    "Final Status": "final status"
}


NUMERIC_FIELDS = [
    "employee code",
    "manager employee code",
    "reviewer employee code",
]

DATE_FIELDS = []


In [122]:
raw_docs = []

# Convert each DF row → raw dict
for idx, row in df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [123]:
normalized_docs

[{'employee code': 1004,
  'employee name': 'Gyanendra Singh',
  'email': 'gyanendras@tataplay.com',
  'manager employee code': 2430,
  'manager name': 'Vishal Sethi',
  'manager email': 'vishal.sethi@tataplay.com',
  'reviewer employee code': 1626,
  'reviewer name': 'Suman Kumar Ghosh',
  'region': 'North',
  'department': 'Technology',
  'sub department': 'Field Engineering and Audit',
  'grade': 'M5',
  'grade level': 'M5a',
  'employee evaluation': 'WSEVAL',
  'employee evaluation status': 'COMPLETED',
  'manager evaluation': 'MGREVAL',
  'manager evaluation status': 'COMPLETED',
  'initiate approval': 'FAPPR',
  'initiate approval status': 'COMPLETED',
  'share document': 'SHRPDOC',
  'share document status': 'COMPLETED',
  'final status': 'DOCUMENT APPROVED'},
 {'employee code': 1034,
  'employee name': 'Monoranjon Dutta',
  'email': 'manoranjand@tataplay.com',
  'manager employee code': 4922,
  'manager name': 'Joydeep Roy',
  'manager email': 'Joydeep.Roy@tataplay.com',
  'rev

In [124]:
dst = db[collection]
result = dst.insert_many(normalized_docs)

## Ddata/Probation Confirmation Report.xls

In [135]:


path="Ddata/Probation Confirmation Report.xls"
collection="probation_confirmation_report"
df=pd.read_excel(path,engine="xlrd")

XLRDError: Unsupported format, or corrupt file: Expected BOF record; found b'MIME-Ver'

## Ddata\8.Leave Transaction With Balance Report_Leave Transaction With Balance Report (3).xlsx

In [22]:
path="Ddata\8.Leave Transaction With Balance Report_Leave Transaction With Balance Report (3).xlsx"
collection="leave_transaction_with_balance_report_leave_transaction_with_balance_report"
df=pd.read_excel(path,skiprows=1)

<>:1: SyntaxWarning: invalid escape sequence '\8'
<>:1: SyntaxWarning: invalid escape sequence '\8'
C:\Users\SaptarshiBanik\AppData\Local\Temp\ipykernel_3332\3302636102.py:1: SyntaxWarning: invalid escape sequence '\8'
  path="Ddata\8.Leave Transaction With Balance Report_Leave Transaction With Balance Report (3).xlsx"


In [23]:
df.columns

Index(['EMPLOYEE_CODE', 'NAME', 'EMPLOYEE_EMAIL_ADDRESS', 'ABSENCE_NAME',
       'START_DATE', 'END_DATE', 'DURATION', 'STATUS_OF_LEAVE',
       'APPROVAL_STATUS', 'MANAGER_EMAIL', 'BALANCE_VAL'],
      dtype='object')

In [24]:
FIELDS = [
    "EMPLOYEE_CODE",
    "NAME",
    "EMPLOYEE_EMAIL_ADDRESS",
    "ABSENCE_NAME",
    "START_DATE",
    "END_DATE",
    "DURATION",
    "STATUS_OF_LEAVE",
    "APPROVAL_STATUS",
    "MANAGER_EMAIL",
    "BALANCE_VAL"
]


KEY_RENAMES = {
    "EMPLOYEE_CODE": "employee code",
    "NAME": "employee name",
    "EMPLOYEE_EMAIL_ADDRESS": "email",
    "ABSENCE_NAME": "absence name",
    "START_DATE": "start date",
    "END_DATE": "end date",
    "DURATION": "duration",
    "STATUS_OF_LEAVE": "status of leave",
    "APPROVAL_STATUS": "approval status",
    "MANAGER_EMAIL": "manager email",
    "BALANCE_VAL": "balance value"
}

NUMERIC_FIELDS = [
    "employee code",
    "duration",
    "balance value"
]

DATE_FIELDS = [
    "start date",
    "end date"
]


In [26]:
raw_docs = []
df = df.where(df.notna(), None)
# Convert each DF row → raw dict
for idx, row in df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [27]:
normalized_docs

[{'employee code': 4598,
  'employee name': 'Bhuvnesh',
  'email': 'Bhuvnesh.Sharma@tataplay.com',
  'absence name': 'Casual Leave',
  'start date': datetime.datetime(2025, 5, 30, 0, 0),
  'end date': datetime.datetime(2025, 5, 30, 0, 0),
  'duration': 1,
  'status of leave': 'SUBMITTED',
  'approval status': 'NA',
  'manager email': 'ravinder.tyagi@tataplay.com',
  'balance value': 11},
 {'employee code': 4598,
  'employee name': 'Bhuvnesh',
  'email': 'Bhuvnesh.Sharma@tataplay.com',
  'absence name': 'Sick Leave   .',
  'start date': datetime.datetime(2025, 5, 9, 0, 0),
  'end date': datetime.datetime(2025, 5, 9, 0, 0),
  'duration': 1,
  'status of leave': 'SUBMITTED',
  'approval status': 'NA',
  'manager email': 'ravinder.tyagi@tataplay.com',
  'balance value': 11},
 {'employee code': 4598,
  'employee name': 'Bhuvnesh',
  'email': 'Bhuvnesh.Sharma@tataplay.com',
  'absence name': 'Annual -Leave',
  'start date': datetime.datetime(2025, 1, 30, 0, 0),
  'end date': datetime.datetim

In [28]:
dst = db[collection]
result = dst.insert_many(normalized_docs)

## Ddata\Goal Detail Report.xlsx

In [43]:
path="Ddata\Goal Detail Report.xlsx"
collection="goal_detail_report"
df=pd.read_excel(path,skiprows=1)

<>:1: SyntaxWarning: invalid escape sequence '\G'
<>:1: SyntaxWarning: invalid escape sequence '\G'
C:\Users\SaptarshiBanik\AppData\Local\Temp\ipykernel_3332\3362400384.py:1: SyntaxWarning: invalid escape sequence '\G'
  path="Ddata\Goal Detail Report.xlsx"


In [44]:
df.columns


Index(['Person Number', 'First Name', 'Last Name', 'Department',
       'Sub Department', 'Grade', 'Sub Grade', 'Business Unit', 'Goal Type',
       'Goal Name', 'Goal Desciption', 'Start Date', 'Target Completion Date',
       'Creation Date', 'Development Goal Status', 'Category',
       'Manager_Approval_Date'],
      dtype='object')

In [45]:
FIELDS = [
    "Person Number",
    "First Name",
    "Last Name",
    "Department",
    "Sub Department",
    "Grade",
    "Sub Grade",
    "Business Unit",
    "Goal Type",
    "Goal Name",
    "Goal Desciption",
    "Start Date",
    "Target Completion Date",
    "Creation Date",
    "Development Goal Status",
    "Category",
    "Manager_Approval_Date"
]

KEY_RENAMES = {
    "Person Number": "employee code",
    "First Name": "first name",
    "Last Name": "last name",
    "Department": "department",
    "Sub Department": "sub department",
    "Grade": "grade",
    "Sub Grade": "grade level",
    "Business Unit": "region",
    "Goal Type": "goal type",
    "Goal Name": "goal name",
    "Goal Desciption": "goal description",
    "Start Date": "start date",
    "Target Completion Date": "target completion date",
    "Creation Date": "creation date",
    "Development Goal Status": "development goal status",
    "Category": "category",
    "Manager_Approval_Date": "manager approval date"
}

NUMERIC_FIELDS = [
    "employee code",
]

DATE_FIELDS = [
    "start date",
    "target completion date",
    "creation date",
    "manager approval date"
]



In [46]:
raw_docs = []
df = df.where(df.notna(), None)
# Convert each DF row → raw dict
for idx, row in df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [47]:
normalized_docs

[{'employee code': 1004,
  'first name': 'Gyanendra',
  'last name': 'Singh',
  'department': 'Technology',
  'sub department': 'Field Engineering and Audit',
  'grade': 'M5',
  'grade level': 'M5a',
  'region': 'North',
  'goal type': 'DEVELOPMENT',
  'goal name': 'Anchor Competency - Chase Result',
  'goal description': 'Improvement will be seen in 3P audit reports as wrong reporting cases will be decreased',
  'start date': datetime.datetime(2020, 10, 1, 0, 0),
  'target completion date': datetime.datetime(2021, 3, 31, 0, 0),
  'creation date': datetime.datetime(2020, 10, 14, 5, 54, 27, tzinfo=datetime.timezone.utc),
  'development goal status': 'NOT_STARTED',
  'category': '70% - On the Job learning',
  'manager approval date': None},
 {'employee code': 1034,
  'first name': 'Monoranjon',
  'last name': 'Dutta',
  'department': 'Field Service Delivery',
  'sub department': 'Field Service Delivery - General',
  'grade': 'M4',
  'grade level': 'M4a',
  'region': 'East',
  'goal type'

In [48]:
dst = db[collection]
result = dst.insert_many(normalized_docs)

## Ddata/Output1.xls
### Offboarding Checklist

In [77]:
path="Ddata/Output1.xls"
collection="offboarding_checklist2"
df = pd.read_excel(path, header=[0, 1])

In [78]:
for i in df.columns:
    print(i)

('Core HR', 'Employee ID')
('Core HR', 'Employee name')
('Core HR', 'Grade')
('Core HR', 'Grade Level')
('Core HR', 'Designation')
('Core HR', 'Department')
('Core HR', 'Sub Department')
('Core HR', 'Region')
('Core HR', 'Employee Status')
('Core HR', 'Reporting Manager')
('Core HR', 'Date of  Joining')
('Core HR', 'Date of Resignation')
('Core HR', 'Date of Leaving')
('Status', ' Exit Checklist - Employee')
('Status', 'Exit Checklist - Reporting Manager')
('Status', 'Exit Checklist - Customer Operations')
('Status', 'Exit Checklist - Facilities')
('Status', 'Exit Checklist - IT')
('Status', 'Exit Checklist - Finance Accounts')
('Status', 'Exit Checklist - Finance Assets')
('Status', 'Exit Checklist - RHR Ops')
('Status', 'Exit Checklist - Payroll')
('Status', 'Exit Checklist - RHR')
('Status', 'Exit Checklist - CHR Ops')
('Status', 'Exit Checklist - CHR')
('Status', 'All Task Status')
(' Exit Checklist - Employee', 'Pending Flexi pay Claims sent to payroll')
(' Exit Checklist - Employ

In [ ]:
FIELDS = {
    "Core HR": [
        "Employee ID",
        "Employee name",
        "Grade",
        "Grade Level",
        "Designation",
        "Department",
        "Sub Department",
        "Region",
        "Employee Status",
        "Reporting Manager",
        "Date of  Joining",
        "Date of Resignation",
        "Date of Leaving",
    ],

    "Status": [
        " Exit Checklist - Employee",
        "Exit Checklist - Reporting Manager",
        "Exit Checklist - Customer Operations",
        "Exit Checklist - Facilities",
        "Exit Checklist - IT",
        "Exit Checklist - Finance Accounts",
        "Exit Checklist - Finance Assets",
        "Exit Checklist - RHR Ops",
        "Exit Checklist - Payroll",
        "Exit Checklist - RHR",
        "Exit Checklist - CHR Ops",
        "Exit Checklist - CHR",
        "All Task Status",
    ],

    "Exit Checklist - Employee": [
        "Pending Flexi pay Claims sent to payroll",
        "Proof of Investements sent to Payroll",
        "Travel claims raised and Manager has approved it in system",
        "Pending Hotel bills sent to Osource",
        "Personal Email ID/Address",
    ],

    "Exit Checklist - Reporting Manager": [
        "Outstanding matters completed",
        "Material Returned",
        "Password and documents handed over",
        "All expenses and Claims approved",
        "Comments",
    ],

    "Exit Checklist - Customer Operations": [
        "Employee Account Disabled",
    ],

    "Exit Checklist - Facilities": [
        "Employee ID Card returned",
        "Access Card Returned",
        "Mention the Amount in case of recovery for Employee",
        "Stationary Returned",
        "Keys to workstation/locker returned",
        "Business card returned",
        "Corporate CC Returned",
        "COCP Recovery Amount (Mobile)",
        "COCP Recovery Amount (Broadband)",
        "Comments",
    ],

    "Exit Checklist - IT": [
        "PC/Laptop Returned",
        "Amount of Recovery for PC/Laptop",
        "Cosummables/Accossories returned",
        "Amount of recovery for consumables/accessories",
        "Account Ids (login id.,email id.)/SIEBEL/SAP/ESS password disabled",
        "Temp Network folders deleted",
        "CRM(SSO)/PRM/Kenan/VMS/EVD/Victory/BO/SAP IDs Disabled",
        "NAS Folder Name with Access Details/Other disabled",
        "Comments",
    ],

    "Exit Checklist - Finance Accounts": [
        "Outstanding Travel Advance/Imprest amount to be recovered",
        "Outstanding relocation amount to be recovered",
        "Travel claim bill received",
        "Amount of recovery for Travel",
        "Personal Expense",
        "Comments",
    ],

    "Exit Checklist - Finance Assets": [
        "Assets submitted by employee",
        "Amount to be recovered against asset",
        "Comments",
    ],

    "Exit Checklist - RHR Ops": [
        "Notice Period Recovery",
        "Verified all checklist data submitted by respective department",
        "DD received Notice recovery",
        "Any leave without pay not applied in the system",
        "Recovery Amount (except -JB,NP,Relocation)",
        "Comments",
    ],

    "Exit Checklist - RHR": [
        "Employee Status",
        "Comments",
    ],

    "Exit Checklist - CHR Ops": [
        "Amount of Training Cost to be recovered",
        "Notice Buy-Out Reimbursement Cost to be Recovered",
        "Joining Bonus to be recovered",
        "Annual Leave Balance (In days)",
        "Encashable Sick Leave Balance (In days)",
        "Encashable Casual Leave Balance (In days)",
        "Special Leave Deduction",
        "Group Transfer",
        "Group Compnay Name",
        "DoJ in Group Company",
        "Gratuity Processed from the Group Company, if any",
        "Tata Group DoJ (If different from date mentioned earlier)",
        "Name of the Point of Contact  of the new Group Company",
        "Designation of the POC",
        "Department of the POC",
        "Email ID of the POC",
        "Contact Number of the POC",
        "Annual Leave Transfer to new Group Company",
        "Number of days of leaves to be transferred",
    ],

    "Exit Checklist - CHR": [
        "Comments",
    ],

    "Exit Checklist - Payroll": [
        "Gratuity Eligibility",
        "Gratuity Payable",
        "TataPlay Subscription",
        "Performance linked Incentive (Sales Incentive)",
        "Performance Linked Incentive (Bonus)",
        "Any other payout (LTIP / Special allowance /other)",
        "Rewards",
        "Statutory Bonus",
        "Relocation reimbursement",
        "City Compensatory Allowance",
        "MIP Deduct/Refund",
        "MIP Top up Deduction/Refund",
        "NPS Recovery",
        "PF Recovery",
    ]
}


KEY_RENAMES = {

    "Core HR": {
        "Employee ID": "employee code",
        "Employee name": "employee name",
        "Grade": "grade",
        "Grade Level": "grade level",
        "Designation": "designation",
        "Department": "department",
        "Sub Department": "sub department",
        "Region": "region",
        "Employee Status": "employee status",
        "Reporting Manager": "manager name",
        "Date of  Joining": "date of joining",
        "Date of Resignation": "date of resignation",
        "Date of Leaving": "date of leaving",
    },

    "Status": {
        " Exit Checklist - Employee": "exit checklist employee",
        "Exit Checklist - Reporting Manager": "exit checklist reporting manager",
        "Exit Checklist - Customer Operations": "exit checklist customer operations",
        "Exit Checklist - Facilities": "exit checklist facilities",
        "Exit Checklist - IT": "exit checklist it",
        "Exit Checklist - Finance Accounts": "exit checklist finance accounts",
        "Exit Checklist - Finance Assets": "exit checklist finance assets",
        "Exit Checklist - RHR Ops": "exit checklist rhr ops",
        "Exit Checklist - Payroll": "exit checklist payroll",
        "Exit Checklist - RHR": "exit checklist rhr",
        "Exit Checklist - CHR Ops": "exit checklist chr ops",
        "Exit Checklist - CHR": "exit checklist chr",
        "All Task Status": "all task status",
    },

    "Exit Checklist - Employee": {
        "Pending Flexi pay Claims sent to payroll": "pending flexi pay claims sent to payroll",
        "Proof of Investements sent to Payroll": "proof of investments sent to payroll",
        "Travel claims raised and Manager has approved it in system": "travel claims approved by manager",
        "Pending Hotel bills sent to Osource": "pending hotel bills sent to osource",
        "Personal Email ID/Address": "personal email",
    },

    "Exit Checklist - Reporting Manager": {
        "Outstanding matters completed": "outstanding matters completed",
        "Material Returned": "material returned",
        "Password and documents handed over": "password and documents handed over",
        "All expenses and Claims approved": "all expenses and claims approved",
        "Comments": "comments",
    },

    "Exit Checklist - Customer Operations": {
        "Employee Account Disabled": "employee account disabled",
    },

    "Exit Checklist - Facilities": {
        "Employee ID Card returned": "employee id card returned",
        "Access Card Returned": "access card returned",
        "Mention the Amount in case of recovery for Employee": "recovery amount employee",
        "Stationary Returned": "stationary returned",
        "Keys to workstation/locker returned": "keys workstation/locker returned",
        "Business card returned": "business card returned",
        "Corporate CC Returned": "corporate cc returned",
        "COCP Recovery Amount (Mobile)": "cocp recovery amount mobile",
        "COCP Recovery Amount (Broadband)": "cocp recovery amount broadband",
        "Comments": "comments",
    },

    "Exit Checklist - IT": {
        "PC/Laptop Returned": "pc/laptop returned",
        "Amount of Recovery for PC/Laptop": "amount recovery pc/laptop",
        "Cosummables/Accossories returned": "consumables accessories returned",
        "Amount of recovery for consumables/accessories": "amount recovery consumables/accessories",
        "Account Ids (login id.,email id.)/SIEBEL/SAP/ESS password disabled": "account ids (login id.,email id.)/siebel/sap/ess password disabled",
        "Temp Network folders deleted": "temp network folders deleted",
        "CRM(SSO)/PRM/Kenan/VMS/EVD/Victory/BO/SAP IDs Disabled": "crm(sso)/prm/kenan/vms/evd/victory/bo/sap ids disabled",
        "NAS Folder Name with Access Details/Other disabled": "nas folder access disabled",
        "Comments": "comments",
    },

    "Exit Checklist - Finance Accounts": {
        "Outstanding Travel Advance/Imprest amount to be recovered": "outstanding travel advance/imprest recovered",
        "Outstanding relocation amount to be recovered": "outstanding relocation amount recovered",
        "Travel claim bill received": "travel claim bill received",
        "Amount of recovery for Travel": "amount recovery travel",
        "Personal Expense": "personal expense",
        "Comments": "comments",
    },

    "Exit Checklist - Finance Assets": {
        "Assets submitted by employee": "assets submitted employee",
        "Amount to be recovered against asset": "amount to be recovered against asset",
        "Comments": "comments",
    },

    "Exit Checklist - RHR Ops": {
        "Notice Period Recovery": "notice period recovery",
        "Verified all checklist data submitted by respective department": "verified all checklist data submitted by respective department",
        "DD received Notice recovery": "dd received notice recovery",
        "Any leave without pay not applied in the system": "any leave without pay not applied in the system",
        "Recovery Amount (except -JB,NP,Relocation)": "recovery amount general except jb np relocation",
        "Comments": "comments",
    },

    "Exit Checklist - RHR": {
        "Employee Status": "employee status",
        "Comments": "comments",
    },

    "Exit Checklist - CHR Ops": {
        "Amount of Training Cost to be recovered": "training cost recovered",
        "Notice Buy-Out Reimbursement Cost to be Recovered": "notice buyout reimbursement recovered",
        "Joining Bonus to be recovered": "joining bonus recovered",
        "Annual Leave Balance (In days)": "annual leave balance days",
        "Encashable Sick Leave Balance (In days)": "sick leave balance encashable in days",
        "Encashable Casual Leave Balance (In days)": "casual leave balance encashable in days",
        "Special Leave Deduction": "special leave deduction",
        "Group Transfer": "group transfer",
        "Group Compnay Name": "group company name",
        "DoJ in Group Company": "date of joining group company",
        "Gratuity Processed from the Group Company, if any": "gratuity processed group company",
        "Tata Group DoJ (If different from date mentioned earlier)": "tata group date of joining",
        "Name of the Point of Contact  of the new Group Company": "poc name group company",
        "Designation of the POC": "poc designation",
        "Department of the POC": "poc department",
        "Email ID of the POC": "poc email",
        "Contact Number of the POC": "poc contact number",
        "Annual Leave Transfer to new Group Company": "annual leave transfer group company",
        "Number of days of leaves to be transferred": "leave transfer days",
    },

    "Exit Checklist - CHR": {
        "Comments": "comments",
    },

    "Exit Checklist - Payroll": {
        "Gratuity Eligibility": "gratuity eligibility",
        "Gratuity Payable": "gratuity payable",
        "TataPlay Subscription": "tataplay subscription",
        "Performance linked Incentive (Sales Incentive)": "performance linked incentive sales incentive",
        "Performance Linked Incentive (Bonus)": "performance linked incentive bonus incentive",
        "Any other payout (LTIP / Special allowance /other)": "other payout",
        "Rewards": "rewards",
        "Statutory Bonus": "statutory bonus",
        "Relocation reimbursement": "relocation reimbursement",
        "City Compensatory Allowance": "city compensatory allowance",
        "MIP Deduct/Refund": "mip deduct/refund",
        "MIP Top up Deduction/Refund": "mip topup deduct/refund",
        "NPS Recovery": "nps recovery",
        "PF Recovery": "pf recovery",
    }
}



NUMERIC_FIELDS = {
    "Core HR": ["employee code"],
    "Exit Checklist - Facilities": [
        "cocp recovery amount mobile",
        "cocp recovery amount broadband"
    ],
    "Exit Checklist - IT": [
        "amount recovery pc/laptop",
        "amount recovery consumables/accessories"
    ],
    "Exit Checklist - Finance Accounts": [
        "outstanding travel advance/imprest recovered",
        "outstanding relocation amount recovered",
        "amount recovery travel",
        "personal expense"
    ],
    "Exit Checklist - Finance Assets": [
        "amount to be recovered against asset"
    ],
    "Exit Checklist - RHR Ops": [
        "dd received notice recovery",
        "Recovery Amount (except -JB,NP,Relocation)"
    ],
    "Exit Checklist - CHR Ops": [
        "annual leave balance days",
        "sick leave balance encashable in days",
        "casual leave balance encashable in days",
        "leave transfer days"
    ],
    "Exit Checklist - Payroll": [
        "gratuity payable",
        "tataplay subscription",
        "performance linked incentive sales incentive",
        "performance linked incentive bonus incentive",
        "other payout",
        "rewards",
        "statutory bonus",
        "relocation reimbursement",
        "city compensatory allowance",
        "mip deduct/refund",
        "mip topup deduct/refund",
        "nps recovery",
        "pf recovery",
    ]
}

DATE_FIELDS = {
    "Core HR": [
        "date of joining",
        "date of resignation",
        "date of leaving"
    ],
    "Exit Checklist - CHR Ops": [
        "date of joining group company",
        "tata group date of joining"
    ]
}

In [80]:
def normalize_nested_row(row, fields, key_renames, numeric_fields, date_fields):
    """
    row: pandas Series where row[key] is accessed with multi-level keys
         e.g., row["Core HR"]["Grade"]
    fields: dict of section → list of fields
    key_renames: dict of section → dict of old→new names
    numeric_fields: dict of section → set of numeric field names
    date_fields: dict of section → set of date field names
    """

    cleaned_sections = {}

    for section, field_list in fields.items():
        section_cleaned = {}

        for field in field_list:
            raw_value = None

            # Retrieve nested value if section exists
            try:
                raw_value = row[(section, field)]
            except Exception:
                raw_value = None

            # Rename field inside the section
            new_field = key_renames.get(section, {}).get(field, field)

            # Determine numeric/date membership
            section_numeric = numeric_fields.get(section, set())
            section_dates = date_fields.get(section, set())

            # Build a fake 1D raw_doc to reuse normalize_doc_generic
            fake_raw = {field: raw_value}

            cleaned_value = normalize_doc_generic(
                fake_raw,
                fields=[field],
                key_renames={field: new_field},
                numeric_fields=section_numeric,
                date_fields=section_dates
            )[new_field]

            section_cleaned[new_field] = cleaned_value

        cleaned_sections[section] = section_cleaned

    return cleaned_sections


In [81]:
# Ensure pandas sees missing values as None
df = df.where(df.notna(), None)

normalized_docs = []

for _, row in df.iterrows():
    nested_cleaned = normalize_nested_row(
        row=row,
        fields=FIELDS,
        key_renames=KEY_RENAMES,
        numeric_fields=NUMERIC_FIELDS,
        date_fields=DATE_FIELDS
    )
    normalized_docs.append(nested_cleaned)


In [82]:
normalized_docs

[{'Core HR': {'employee code': 2220,
   'employee name': 'Anshul Kapoor',
   'grade': 'M3',
   'grade level': 'M3b',
   'designation': 'Assistant General Manager - Facilities',
   'department': 'Facilities',
   'sub department': 'Facilities - General',
   'region': 'North',
   'employee status': 'Active',
   'manager name': 'Chanmeet Singh',
   'date of joining': datetime.datetime(2007, 1, 2, 0, 0),
   'date of resignation': datetime.datetime(2025, 8, 4, 0, 0),
   'date of leaving': datetime.datetime(2025, 11, 3, 0, 0)},
  'Status': {'exit checklist employee': 'Pending',
   'exit checklist reporting manager': 'Pending',
   'exit checklist customer operations': 'Pending',
   'exit checklist facilities': 'Pending',
   'exit checklist it': 'Pending',
   'exit checklist finance accounts': 'Pending',
   'exit checklist finance assets': 'Pending',
   'exit checklist rhr ops': 'Pending',
   'exit checklist payroll': 'Pending',
   'exit checklist rhr': 'Pending',
   'exit checklist chr ops': '

In [71]:
df.head

<bound method NDFrame.head of        Core HR                                                   \
   Employee ID                  Employee name Grade Grade Level   
0         2220                  Anshul Kapoor    M3         M3b   
1          245  Donthamsetti Venkata Subbarao    M4         M4a   
2         3893                  Amit Paralkar    M3         M3a   
3         3558            Seshadri Manivannan    M2         M2a   
4         1361               Ashok Kumar M.K.    M4         M4b   
..         ...                            ...   ...         ...   
72        7966                  Somesh Sharma    M4         M4b   
73        7987                Swapnil Bhosale    M3         M3a   
74        7989                    Vimal Kumar    M4         M4b   
75        8000             Harshvardhan Verma    M5         M5a   
76        8035                 Keshav Kaushik    M5         M5a   

                                                                          \
                      

In [83]:
dst = db[collection]
result = dst.insert_many(normalized_docs)

# Old Files Data Cleaning Pipeline Generation

## Goal Status Report 2025-26 All.xlsx aka Goal Setting Status

In [22]:
path="../Ddata/Goal Status Report 2025-26 All.xlsx"
collection="goal_setting_status"
df=pd.read_excel(path,skiprows=1)

In [23]:
df.columns

Index(['PERSON_NUMBER', 'EMP_NAME', 'Department', 'SUB_DEPARTMENT',
       'Designation', 'Office_Location', 'Work_location', 'Parent_Grade',
       'Sub_grade', 'Region', 'DOJ', 'Manager_Number', 'Reporting_To',
       'REVIEWER_NUMBER', 'REVIEWER_NAME', 'STATUS'],
      dtype='object')

In [ ]:
FIELDS = [
    'PERSON_NUMBER', 
    'EMP_NAME', 
    'Department', 
    'SUB_DEPARTMENT',
    'Designation', 
    'Office_Location', 
    'Work_location', 
    'Parent_Grade',
    'Sub_grade', 
    'Region', 
    'DOJ', 
    'Manager_Number',
    'Reporting_To',
    'REVIEWER_NUMBER', 
    'REVIEWER_NAME', 
    'STATUS']


KEY_RENAMES = {
    "PERSON_NUMBER" : "employee code", 
    "EMP_NAME" : "employee name", 
    "Department" : "department", 
    "SUB_DEPARTMENT" : "sub department",
    "Designation" : "designation", 
    "Office_Location" : "office", 
    "Work_location" : "location", 
    "Parent_Grade" : "grade",
    "Sub_grade" : "grade level", 
    "Region" : "region", 
    "DOJ" : "date of joining", 
    "Manager_Number" : "manager employee code",
    "Reporting_To" : "reporting to",
    "REVIEWER_NUMBER" : "reviewer employee code", 
    "REVIEWER_NAME" : "reviewer name", 
    "STATUS":"status"
}

NUMERIC_FIELDS = [
    "employee code",
    "manager employee code",
    "reviewer employee code"
]

DATE_FIELDS = ["date of joining"]


In [25]:
raw_docs = []
df = df.where(df.notna(), None)
# Convert each DF row → raw dict
for idx, row in df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [28]:
normalized_docs

[{'employee code': 1004,
  'employee name': 'Gyanendra Singh',
  'department': 'Technology',
  'sub department': 'Field Engineering and Audit',
  'designation': 'Assistant Manager - Quality',
  'office': 'Lucknow',
  'location': 'Lucknow',
  'grade': 'M5',
  'grade level': 'M5a',
  'region': 'North',
  'date of joining': datetime.datetime(2006, 4, 3, 0, 0),
  'manager employee code': 2430,
  'reporting to': 'Vishal Sethi',
  'reviewer employee code': 1626,
  'reviewer name': 'Suman Kumar Ghosh',
  'STATUS': 'APPROVED'},
 {'employee code': 1034,
  'employee name': 'Monoranjon Dutta',
  'department': 'Field Service Delivery',
  'sub department': 'Field Service Delivery - General',
  'designation': 'Senior Manager - Field Service Delivery',
  'office': 'Kolkata',
  'location': 'Kolkata',
  'grade': 'M4',
  'grade level': 'M4a',
  'region': 'East',
  'date of joining': datetime.datetime(2006, 4, 10, 0, 0),
  'manager employee code': 4922,
  'reporting to': 'Joydeep Roy',
  'reviewer employ

## Performance Goal Report 25-26.xlsx

In [35]:
path="../Ddata/Performance Goal Report 25-26.xlsx"
collection="goal_setting_status"
df=pd.read_excel(path,skiprows=2)

In [39]:
df.columns

Index(['Person Number', 'Name', 'Department', 'Sub Department', 'Grade',
       'Sub Grade', 'Region', 'Review Period Name', 'Goal Plan Name',
       'Goal Name', 'Weight', 'Description'],
      dtype='object')

In [40]:
FIELDS = ['Person Number', 'Name', 'Department', 'Sub Department', 'Grade',
       'Sub Grade', 'Region', 'Review Period Name', 'Goal Plan Name',
       'Goal Name', 'Weight', 'Description']


KEY_RENAMES = {
        "Person Number" : "employee code",
        "Name" : "employee name",
        "Department" : "Department",
        "Sub Department" : "sub department",
        "Grade" : "grade",
        "Sub Grade" : "grade level",
        "Region" : "region",
        "Review Period Name" : "review period name",
        "Goal Plan Name" : "goal plan name",
        "Goal Name" : "goal name",
        "Weight" : "weight",
        "Description" : "description"
}

NUMERIC_FIELDS = [
    "employee code",
    "weight"
]

DATE_FIELDS = []


In [41]:
raw_docs = []
df = df.where(df.notna(), None)
# Convert each DF row → raw dict
for idx, row in df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [42]:
normalized_docs

[{'employee code': 1004,
  'employee name': 'Gyanendra Singh',
  'Department': 'Technology',
  'sub department': 'Field Engineering and Audit',
  'grade': 'M5',
  'grade level': 'M5a',
  'region': 'North',
  'review period name': 'Performance Review 2025-26',
  'goal plan name': 'Goal Plan for 2025-26',
  'goal name': 'Buddy Ride, Competiton Audit & Field Trial Audits',
  'weight': 20,
  'description': '* Buddy Ride Activity of Third Party Auditors - Each Auditor Should be covered every month in base city * Special Task as per requirement * Audit of Field Trial Material as per requirement'},
 {'employee code': 1004,
  'employee name': 'Gyanendra Singh',
  'Department': 'Technology',
  'sub department': 'Field Engineering and Audit',
  'grade': 'M5',
  'grade level': 'M5a',
  'region': 'North',
  'review period name': 'Performance Review 2025-26',
  'goal plan name': 'Goal Plan for 2025-26',
  'goal name': 'MDU Audits',
  'weight': 10,
  'description': '* Conducting MDU Capex Audits (>=

## 1.DatabaseReport_Rpt_Data base Report_for Kreeda Labs.xlsx aka Base Report

In [50]:
path="../Ddata/1.DatabaseReport_Rpt_Data base Report_for Kreeda Labs.xlsx"
collection="goal_setting_status"
df=pd.read_excel(path,skiprows=0)

In [51]:
df.columns

Index(['Employee Code', 'FIRST NAME', 'LAST NAME', 'GRADE', 'Grade Level',
       'Designation', 'DEPARTMENT', 'SUB-DEPT', 'Location', 'Office', 'Region',
       'DOJ', 'Manager Code', 'Reporting To', 'State', 'Circle', 'DOB', 'M/F',
       'Primary Email', 'Assignment Status Type', 'DOR', 'DOL', 'Role',
       'Last year Rating', 'Reason for Resignation'],
      dtype='object')

In [52]:
FIELDS = ['Employee Code', 'FIRST NAME', 'LAST NAME', 'GRADE', 'Grade Level',
       'Designation', 'DEPARTMENT', 'SUB-DEPT', 'Location', 'Office', 'Region',
       'DOJ', 'Manager Code', 'Reporting To', 'State', 'Circle', 'DOB', 'M/F',
       'Primary Email', 'Assignment Status Type', 'DOR', 'DOL', 'Role',
       'Last year Rating', 'Reason for Resignation']

KEY_RENAMES = {
        "Employee Code" : "employee code",
        "FIRST NAME" : "first name",
        "LAST NAME" : "last name",
        "GRADE" : "grade",
        "Grade Level" : "grade level",
        "Designation" : "Designation",
        "DEPARTMENT" : "department",
        "SUB-DEPT" : "sub department",
        "Location" : "location",
        "Office" : "office",
        "Region" : "region",
        "DOJ" : "date of joining",
        "Manager Code" : "manager employee code",
        "Reporting To" : "reporting to",
        "State" : "state",
        "Circle" : "circle",
        "DOB" : "date of birth",
        "M/F" : "gender",
        "Primary Email" : "email",
        "Assignment Status Type" : "assignment status type",
        "DOR" : "date of resignation",
        "DOL" : "date of leaving",
        "Role" : "role",
        "Last year Rating" : "last year rating",
        "Reason for Resignation" : "reason for resignation"
}

NUMERIC_FIELDS = [
    "employee code",
    "manager employee code",
    "last year rating"
]

DATE_FIELDS = ["date of birth","date of joining","date of resignation","date of leaving",]


In [53]:
raw_docs = []
df = df.where(df.notna(), None)
# Convert each DF row → raw dict
for idx, row in df.iterrows():
    raw_doc = row.to_dict()


    raw_docs.append(raw_doc)

# Normalize
normalized_docs = [
    normalize_doc_generic(
        doc,
        FIELDS,
        KEY_RENAMES,
        NUMERIC_FIELDS,
        DATE_FIELDS
    )
    for doc in raw_docs
]


In [54]:
normalized_docs

[{'employee code': 1,
  'first name': 'Vikram',
  'last name': 'Kaushik',
  'grade': 'M0',
  'grade level': 'M.0',
  'Designation': 'Managing Director and CEO',
  'department': 'Executive Office',
  'sub department': 'Executive Office - General',
  'location': 'Mumbai',
  'office': 'Corporate',
  'region': 'Corporate',
  'date of joining': datetime.datetime(2004, 3, 22, 0, 0),
  'manager employee code': None,
  'reporting to': 'NA',
  'state': 'Maharashtra',
  'circle': 'NA',
  'date of birth': datetime.datetime(1950, 9, 17, 0, 0),
  'gender': 'Male',
  'email': 'vikramk@tatasky.com',
  'assignment status type': 'INACTIVE',
  'date of resignation': datetime.datetime(2010, 12, 31, 0, 0),
  'date of leaving': datetime.datetime(2010, 12, 31, 0, 0),
  'role': 'NA',
  'last year rating': None,
  'reason for resignation': 'NA'},
 {'employee code': 10,
  'first name': 'Irene',
  'last name': "D'Cunha",
  'grade': 'M4',
  'grade level': 'M4b',
  'Designation': "Manager CEO's Office",
  'depart